# PCB Defect Detection — YOLOv11s Colab Training

Fine-tunes `yolo11s.pt` on the PKU-Market-PCB dataset (6 defect classes, YOLO format,
prepared by `python -m pcb_vision.data_prep`).

**How to run**
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`
3. When prompted, upload `pcb_yolo.zip` (create it locally with `python scripts/make_dataset_zip.py`).
   Or press *Cancel* and the notebook falls back to `MyDrive/pcb_yolo.zip` in your Google Drive.
4. Leave the tab open — training takes roughly 1.5–3 h on a T4 (early stopping, `patience=25`,
   may finish sooner). When done, `pcb_yolo11s_run.zip` downloads automatically —
   unzip it into the repo's `models/` folder.

Every cell is idempotent: re-running all cells skips work that is already done.

In [ ]:
# 1 — GPU check + install Ultralytics
!nvidia-smi
%pip install -q ultralytics

In [ ]:
# 2 — Get the dataset zip (upload, or fall back to Google Drive) and unzip
import shutil
import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/pcb_yolo.zip")
DATA_DIR = Path("/content/pcb_yolo")

if not DATA_DIR.exists():
    if not ZIP_PATH.exists():
        uploaded = {}
        try:
            from google.colab import files
            print("Select pcb_yolo.zip (press Cancel to use Drive: MyDrive/pcb_yolo.zip)")
            uploaded = files.upload()
        except Exception as exc:  # upload widget unavailable or cancelled
            print(f"Upload skipped ({exc}) — falling back to Google Drive")
        zips = [n for n in uploaded if n.endswith(".zip")]
        if zips:
            if Path(zips[0]).resolve() != ZIP_PATH:
                shutil.move(zips[0], ZIP_PATH)
        else:
            from google.colab import drive
            drive.mount("/content/drive")
            drive_zip = Path("/content/drive/MyDrive/pcb_yolo.zip")
            assert drive_zip.exists(), "Not found: MyDrive/pcb_yolo.zip — upload it to your Drive root"
            shutil.copy(drive_zip, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall("/content")

counts = {s: len(list((DATA_DIR / "images" / s).glob("*"))) for s in ("train", "val", "test")}
print(f"Images per split: {counts}")
assert (DATA_DIR / "dataset.yaml").is_file() and counts["train"] > 0, "Dataset extraction failed"

In [ ]:
# 3 — Point dataset.yaml at the Colab copy
import yaml

cfg_path = DATA_DIR / "dataset.yaml"
cfg = yaml.safe_load(cfg_path.read_text())
cfg["path"] = str(DATA_DIR)
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg_path.read_text())

In [ ]:
# 4 — Train (skips if a finished run exists; delete /content/runs to retrain)
from pathlib import Path

from ultralytics import YOLO

RUNS = Path("/content/runs")
BEST = RUNS / "pcb_yolo11s" / "weights" / "best.pt"

if BEST.exists():
    print("best.pt already exists — skipping training")
else:
    YOLO("yolo11s.pt").train(
        data=str(DATA_DIR / "dataset.yaml"),
        epochs=120,
        imgsz=1024,
        batch=8,
        patience=25,
        seed=42,
        project=str(RUNS),
        name="pcb_yolo11s",
        exist_ok=True,
    )
assert BEST.exists(), "Training did not produce best.pt"

In [ ]:
# 5 — Evaluate on the held-out test split
model = YOLO(str(BEST))
metrics = model.val(
    data=str(DATA_DIR / "dataset.yaml"),
    split="test",
    imgsz=1024,
    project=str(RUNS),
    name="pcb_yolo11s_test",
    exist_ok=True,
)
print(f"TEST  mAP50: {metrics.box.map50:.4f}   mAP50-95: {metrics.box.map:.4f}")

In [ ]:
# 6 — Bundle weights + metrics and download
import zipfile

OUT = Path("/content/pcb_yolo11s_run.zip")
run_dir = RUNS / "pcb_yolo11s"
test_dir = RUNS / "pcb_yolo11s_test"

with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(BEST, "best.pt")
    for name in ("results.csv", "args.yaml"):
        zf.write(run_dir / name, name)
    for img in sorted(run_dir.glob("*.png")) + sorted(run_dir.glob("*.jpg")):
        zf.write(img, img.name)
    if test_dir.exists():
        for img in sorted(test_dir.glob("*.png")):
            zf.write(img, f"test/{img.name}")

print(f"{OUT} — {OUT.stat().st_size / 1e6:.1f} MB")
try:
    from google.colab import files
    files.download(str(OUT))
except Exception:
    print("Not running in Colab — download the zip manually from the file browser")